#### Imports

In [ ]:
import csv
import json
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

#### Configuration

Change only these two values for a run: `DATA` and `COHORT`.
Everything else (paths, CUIs, horizons, first-contact lookup) is derived.

In [ ]:
DATA = "ed"          # "nph" | "pituitary" | "ed"

# ED is inference-only: no treatment/control gating, always drop target CUIs.
if DATA == "ed":
    COHORT = "inference"
else:
    COHORT = "treatment"  # "treatment" | "control" (ignored when DATA == "ed")

MIN_DATE = "2000-01-01"
FIRST_CONTACT_CSV = Path("patient_first_contact.csv")

# Single source of truth for horizon options used across processing and validation.
# - "baseline": no horizon-day cutoff
# - int:        days before anchor (treatment: diagnosis, control/inference: last event)
# - "first_contact": cut to on/before patient's first clinical contact date
HORIZON_OPTIONS = ["baseline", 30, 90, 365, "first_contact"]

print(f"DATA={DATA} COHORT={COHORT}")
print(f"MIN_DATE={MIN_DATE}")
print(f"FIRST_CONTACT_CSV={FIRST_CONTACT_CSV}")
print(f"HORIZON_OPTIONS={HORIZON_OPTIONS}")

In [ ]:
# Derived config: target/excluded CUIs, directory paths.

if DATA == "nph":
    TARGET_CUIS = [230745008, 30753002, 698625002]
    EXCLUDED_CUIS = [
        446541009, 257351008, 424208002, 47020004, 258593008, 277492002,
        1360046000, 88834003, 280392009, 257354000, 700132008, 699006006,
        1360044002, 119566000, 704955004, 681124211000119103,
    ]
    PREFIX = "nph"
elif DATA == "pituitary":
    TARGET_CUIS = [254965007, 254962005, 127024001, 254956000, 399244003]
    EXCLUDED_CUIS = []
    PREFIX = "pa"
elif DATA == "ed":
    # Default ED inference config uses NPH CUI sets.
    # Adjust here if your ED inference target changes.
    TARGET_CUIS = [230745008, 30753002, 698625002]
    EXCLUDED_CUIS = [
        446541009, 257351008, 424208002, 47020004, 258593008, 277492002,
        1360046000, 88834003, 280392009, 257354000, 700132008, 699006006,
        1360044002, 119566000, 704955004, 681124211000119103,
    ]
    PREFIX = "ed"
else:
    raise ValueError(f"Unknown DATA: {DATA}")

TARGET_SET = set(TARGET_CUIS)
EXCLUDED_SET = set(EXCLUDED_CUIS)
overlap = TARGET_SET & EXCLUDED_SET
if overlap:
    raise ValueError(f"TARGET_CUIS and EXCLUDED_CUIS overlap: {overlap}")

if COHORT == "treatment":
    RAW_DIR = Path(f"timelines/{PREFIX}_treatment_timelines_raw")
    CLEAN_DIR = Path(f"timelines/{PREFIX}_treatment_timelines_clean")
elif COHORT == "control":
    RAW_DIR = Path(f"timelines/{PREFIX}_control_timelines_raw")
    CLEAN_DIR = Path(f"timelines/{PREFIX}_control_timelines_clean")
elif COHORT == "inference":
    RAW_DIR = Path(f"timelines/{PREFIX}_timelines_raw")
    CLEAN_DIR = Path(f"timelines/{PREFIX}_timelines_clean")
else:
    raise ValueError(f"Unknown COHORT: {COHORT}")

print(f"TARGET_CUIS   = {TARGET_CUIS}")
print(f"EXCLUDED_CUIS = {EXCLUDED_CUIS}")
print(f"RAW_DIR       = {RAW_DIR} (exists={RAW_DIR.exists()})")
print(f"CLEAN_DIR     = {CLEAN_DIR}")

In [ ]:
# Output directory naming, used by processing AND validation.

def output_dir_for(opt):
    """Return the output dir Path for a given horizon option."""
    if opt == "baseline":
        return CLEAN_DIR
    if opt == "first_contact":
        return CLEAN_DIR.parent / f"{CLEAN_DIR.name}_hfirstcontact"
    if isinstance(opt, int):
        return CLEAN_DIR.parent / f"{CLEAN_DIR.name}_h{opt}"
    raise ValueError(f"Unknown horizon option: {opt!r}")


def label_for(opt):
    if opt == "baseline":
        return "baseline"
    if opt == "first_contact":
        return "hfirstcontact"
    return f"h{opt}"


for opt in HORIZON_OPTIONS:
    print(f"  {label_for(opt):<14} -> {output_dir_for(opt)}")

In [ ]:
# Load first-contact lookup: patient_id (str) -> 'YYYY-MM-DD'

first_contact_by_patient = {}

if not FIRST_CONTACT_CSV.exists():
    print(f"WARNING: {FIRST_CONTACT_CSV} not found. 'first_contact' horizon will skip all patients.")
else:
    with open(FIRST_CONTACT_CSV, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            pid = str(row["patient"]).strip()
            date_str = str(row["date"]).strip()[:10]  # keep date portion only
            first_contact_by_patient[pid] = date_str
    print(f"Loaded first-contact dates for {len(first_contact_by_patient)} patients from {FIRST_CONTACT_CSV}")

In [ ]:
# Discover raw timeline files.

timeline_files = sorted(RAW_DIR.glob("*.json"))
print(f"Found {len(timeline_files)} raw timeline files in {RAW_DIR}")

#### Utility functions

Each utility returns a **new** timeline dict (or `None`) and leaves the input unchanged. `total_entities` and `date_range` are recomputed from the kept events.

In [ ]:
def parse_iso_date(date_str):
    """Parse 'YYYY-MM-DD' (or ISO datetime) into a `datetime.date`."""
    return datetime.fromisoformat(date_str).date()


def _rebuild_timeline(src, kept_events):
    """Shallow-copy `src` and set events + recomputed metadata from `kept_events`."""
    new_t = src.copy()
    new_t["events"] = kept_events
    new_t["total_entities"] = len(kept_events)
    dates = [e["standardized_date"] for e in kept_events if e.get("standardized_date")]
    if dates:
        new_t["date_range"] = {"start": min(dates), "end": max(dates)}
    else:
        new_t.pop("date_range", None)
    return new_t

In [ ]:
def filter_by_date(timeline, min_date):
    """Keep only events on/after `min_date` ('YYYY-MM-DD'). Returns a new dict."""
    kept = [
        e for e in timeline.get("events", [])
        if e.get("standardized_date") and e["standardized_date"] >= min_date
    ]
    return _rebuild_timeline(timeline, kept)


def remove_excluded_cui_events(timeline, excluded_cuis):
    """Drop events whose entity_cui is in `excluded_cuis`. No-op if empty."""
    if not excluded_cuis:
        return timeline
    excluded = set(excluded_cuis)
    kept = [e for e in timeline.get("events", []) if e.get("entity_cui") not in excluded]
    return _rebuild_timeline(timeline, kept)

In [ ]:
def truncate_at_target_cui(timeline, target_cuis):
    """Keep events strictly before the first event with entity_cui in `target_cuis`.

    Returns None if no target CUI is found or the pre-target prefix is empty.
    """
    target = set(target_cuis)
    events = timeline.get("events", [])
    idx = next((i for i, e in enumerate(events) if e.get("entity_cui") in target), None)
    if idx is None:
        return None
    kept = events[:idx]
    if not kept:
        return None
    return _rebuild_timeline(timeline, kept)


def apply_day_horizon(timeline, anchor_date, horizon_days):
    """Keep events with standardized_date <= anchor_date - horizon_days."""
    cutoff = anchor_date - timedelta(days=horizon_days)
    kept = [
        e for e in timeline.get("events", [])
        if e.get("standardized_date")
        and parse_iso_date(e["standardized_date"]) <= cutoff
    ]
    if not kept:
        return None
    return _rebuild_timeline(timeline, kept)


def apply_first_contact_cutoff(timeline, first_contact_date):
    """Keep events with standardized_date <= first_contact_date."""
    kept = [
        e for e in timeline.get("events", [])
        if e.get("standardized_date")
        and parse_iso_date(e["standardized_date"]) <= first_contact_date
    ]
    if not kept:
        return None
    return _rebuild_timeline(timeline, kept)

In [ ]:
# Unified pipeline: raw timeline -> processed timeline for a given horizon option,
# or (None, reason) if the patient is filtered out.
#
# treatment:  pre-diagnosis prefix only, then horizon cutoff anchored at diagnosis.
# control:    must have no target CUI; horizon cutoff anchored at last event.
# inference:  unlabeled mode (ED). No gating. Target CUIs always dropped.
#             horizon cutoff anchored at last event (or first contact date).

def process_timeline(raw_timeline, opt, cohort):
    """Return (processed_timeline or None, skip_reason or None)."""
    # Stage 1: min date filter
    t = filter_by_date(raw_timeline, MIN_DATE)
    if not t["events"]:
        return None, "empty_after_min_date"

    # Stage 2: drop excluded CUIs
    t = remove_excluded_cui_events(t, EXCLUDED_CUIS)
    if not t["events"]:
        return None, "empty_after_excluded_cuis"

    # Stage 3: cohort gating (treatment/control only)
    if cohort in ("treatment", "control"):
        has_target = any(e.get("entity_cui") in TARGET_SET for e in t["events"])
        if cohort == "treatment" and not has_target:
            return None, "no_target_in_treatment"
        if cohort == "control" and has_target:
            return None, "target_in_control"

    # Stage 4: cohort-specific horizon handling
    if cohort == "treatment":
        pre_diag = truncate_at_target_cui(t, TARGET_CUIS)
        if pre_diag is None or not pre_diag["events"]:
            return None, "empty_before_diagnosis"

        if opt == "baseline":
            return pre_diag, None

        if isinstance(opt, int):
            diag_idx = next(
                i for i, e in enumerate(t["events"]) if e.get("entity_cui") in TARGET_SET
            )
            diag_date_str = t["events"][diag_idx].get("standardized_date")
            if not diag_date_str:
                return None, "missing_diagnosis_date"
            out = apply_day_horizon(pre_diag, parse_iso_date(diag_date_str), opt)
            return (out, None) if out else (None, "empty_after_day_horizon")

        if opt == "first_contact":
            fc = first_contact_by_patient.get(str(t.get("patient_id")))
            if not fc:
                return None, "missing_first_contact"
            out = apply_first_contact_cutoff(pre_diag, parse_iso_date(fc))
            return (out, None) if out else (None, "empty_after_first_contact")

        return None, f"unknown_option_{opt}"

    if cohort in ("control", "inference"):
        # For inference, always drop target CUIs (ED outputs must be leakage-free).
        if cohort == "inference" and TARGET_CUIS:
            t = remove_excluded_cui_events(t, TARGET_CUIS)
            if not t["events"]:
                return None, "empty_after_target_drop"

        if opt == "baseline":
            return t, None

        if isinstance(opt, int):
            dates = [
                parse_iso_date(e["standardized_date"])
                for e in t["events"]
                if e.get("standardized_date")
            ]
            if not dates:
                return None, "no_dated_events"
            out = apply_day_horizon(t, max(dates), opt)
            return (out, None) if out else (None, "empty_after_day_horizon")

        if opt == "first_contact":
            fc = first_contact_by_patient.get(str(t.get("patient_id")))
            if not fc:
                return None, "missing_first_contact"
            out = apply_first_contact_cutoff(t, parse_iso_date(fc))
            return (out, None) if out else (None, "empty_after_first_contact")

        return None, f"unknown_option_{opt}"

    return None, f"unknown_cohort_{cohort}"

#### Exploration

Lightweight descriptive stats on the raw timelines to sanity-check inputs. These are optional — safe to skip.

In [ ]:
# Target CUI presence and position

with_target = 0
positions = []
lengths_with = []
lengths_without = []

for fpath in timeline_files:
    with open(fpath, "r") as f:
        timeline = json.load(f)
    events = timeline.get("events", [])
    idxs = [i for i, e in enumerate(events) if e.get("entity_cui") in TARGET_SET]
    n = len(events)
    if idxs:
        with_target += 1
        positions.append((idxs[0] / n) * 100 if n else 0)
        lengths_with.append(n)
    else:
        lengths_without.append(n)

print(f"Timelines with target CUI: {with_target}/{len(timeline_files)}")
if positions:
    print(
        f"First target position (%): mean={np.mean(positions):.1f}, "
        f"min={np.min(positions):.1f}, max={np.max(positions):.1f}"
    )
if lengths_with:
    print(
        f"With-target length:    n={len(lengths_with)}, "
        f"mean={np.mean(lengths_with):.1f}, median={np.median(lengths_with):.1f}"
    )
if lengths_without:
    print(
        f"Without-target length: n={len(lengths_without)}, "
        f"mean={np.mean(lengths_without):.1f}, median={np.median(lengths_without):.1f}"
    )

In [ ]:
# Excluded CUI occurrences in raw timelines

if not EXCLUDED_SET:
    print("EXCLUDED_CUIS is empty — nothing to analyze.")
else:
    per_patient = []
    cui_hits = defaultdict(int)
    with_any = 0
    for fpath in timeline_files:
        with open(fpath, "r") as f:
            timeline = json.load(f)
        events = timeline.get("events", [])
        c = 0
        for e in events:
            cui = e.get("entity_cui")
            if cui in EXCLUDED_SET:
                c += 1
                cui_hits[cui] += 1
        per_patient.append(c)
        if c > 0:
            with_any += 1

    arr = np.array(per_patient, dtype=float)
    print(f"Timelines: {len(timeline_files)}  |  distinct excluded CUIs in config: {len(EXCLUDED_SET)}")
    print(f"Total excluded events: {int(arr.sum())}")
    print(f"Patients with >=1 excluded event: {with_any}/{len(timeline_files)}")
    print(
        f"Per-patient count - mean={arr.mean():.2f}, median={np.median(arr):.1f}, "
        f"min={int(arr.min())}, max={int(arr.max())}"
    )
    print("Top excluded CUIs by event count:")
    for cui, k in sorted(cui_hits.items(), key=lambda x: -x[1])[:10]:
        print(f"  {cui}: {k}")

In [ ]:
# Events before vs after first clinical contact (raw timelines)

if not first_contact_by_patient:
    print("No first-contact lookup loaded — skipping.")
else:
    total_n = 0
    total_before = 0
    missing = 0
    covered = 0
    pct_before = []
    for fpath in timeline_files:
        with open(fpath, "r") as f:
            timeline = json.load(f)
        pid = str(timeline.get("patient_id"))
        fc_str = first_contact_by_patient.get(pid)
        if not fc_str:
            missing += 1
            continue
        fc = parse_iso_date(fc_str)
        dated = [e for e in timeline.get("events", []) if e.get("standardized_date")]
        if not dated:
            continue
        covered += 1
        n = len(dated)
        b = sum(1 for e in dated if parse_iso_date(e["standardized_date"]) < fc)
        total_n += n
        total_before += b
        pct_before.append(100.0 * b / n)

    print(f"Coverage: {covered} covered, {missing} missing first-contact lookup")
    if total_n:
        print(
            f"Event-weighted: before={100.0 * total_before / total_n:.2f}%, "
            f"after/on={100.0 * (total_n - total_before) / total_n:.2f}%"
        )
    if pct_before:
        arr = np.array(pct_before)
        print(f"Per-patient %-before: mean={arr.mean():.2f}, median={np.median(arr):.2f}")

#### Process all horizons

One loop over `HORIZON_OPTIONS` and all raw timelines. Produces all output directories in a single pass with uniform semantics.

In [ ]:
def safe_filename(patient_id):
    pid = str(patient_id).replace("/", "").replace("\\", "")
    return f"patient_{pid}_timeline.json"


saved_counts = {opt: 0 for opt in HORIZON_OPTIONS}
skip_counters = {opt: defaultdict(int) for opt in HORIZON_OPTIONS}

print(f"Processing {len(timeline_files)} raw {COHORT} timelines into {len(HORIZON_OPTIONS)} outputs...")

for opt in HORIZON_OPTIONS:
    out_dir = output_dir_for(opt)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Wipe any stale output from earlier runs so validation can't be tripped up.
    for old in out_dir.glob("*.json"):
        old.unlink()

    for fpath in timeline_files:
        with open(fpath, "r") as f:
            raw = json.load(f)

        processed, reason = process_timeline(raw, opt, COHORT)
        if processed is None:
            skip_counters[opt][reason] += 1
            continue

        fname = safe_filename(processed.get("patient_id"))
        with open(out_dir / fname, "w") as f:
            json.dump(processed, f, indent=2)
        saved_counts[opt] += 1

    print(f"\n[{label_for(opt):<14}] -> {out_dir}")
    print(f"  saved   : {saved_counts[opt]}")
    skipped_total = sum(skip_counters[opt].values())
    print(f"  skipped : {skipped_total}")
    for reason, k in sorted(skip_counters[opt].items(), key=lambda x: -x[1]):
        print(f"    - {reason}: {k}")

print("\nDone processing.")

#### Validation

All validation iterates `HORIZON_OPTIONS` via `output_dir_for`, so every horizon (including `first_contact`) is always covered.

In [ ]:
# Length distribution summary per horizon output dir

for opt in HORIZON_OPTIONS:
    d = output_dir_for(opt)
    if not d.exists():
        print(f"[{label_for(opt)}] missing: {d}")
        continue
    files = sorted(d.glob("*.json"))
    if not files:
        print(f"[{label_for(opt)}] no JSON files in {d}")
        continue
    lengths = []
    for fpath in files:
        with open(fpath, "r") as f:
            timeline = json.load(f)
        lengths.append(len(timeline.get("events", [])))
    arr = np.array(lengths)
    print(f"[{label_for(opt)}] dir={d}")
    print(f"  #timelines: {len(arr)}")
    print(
        f"  events per timeline: mean={arr.mean():.1f}, median={np.median(arr):.1f}, "
        f"min={arr.min()}, max={arr.max()}, p25={np.percentile(arr, 25):.1f}, p75={np.percentile(arr, 75):.1f}"
    )
    print()

In [ ]:
# Forbidden CUI check: no TARGET_CUIS / EXCLUDED_CUIS in any saved event.
# Applies to ALL horizon outputs, including first_contact.

any_failure = False
for opt in HORIZON_OPTIONS:
    d = output_dir_for(opt)
    if not d.exists():
        print(f"[{label_for(opt)}] skip - missing: {d}")
        continue
    files = sorted(d.glob("*.json"))
    if not files:
        print(f"[{label_for(opt)}] skip - no JSON in {d}")
        continue

    bad_target = []
    bad_excluded = []
    for fpath in files:
        with open(fpath, "r") as f:
            timeline = json.load(f)
        for ev in timeline.get("events", []):
            cui = ev.get("entity_cui")
            if cui is None:
                continue
            if cui in TARGET_SET:
                bad_target.append((fpath.name, cui))
            if cui in EXCLUDED_SET:
                bad_excluded.append((fpath.name, cui))

    print(f"[{label_for(opt)}] files={len(files)} dir={d}")
    if bad_target:
        any_failure = True
        print(f"  FAIL - {len(bad_target)} event(s) with TARGET_CUI (showing up to 10):")
        for name, cui in bad_target[:10]:
            print(f"    {name}  entity_cui={cui}")
    else:
        print("  OK - no TARGET_CUIS in events")
    if bad_excluded:
        any_failure = True
        print(f"  FAIL - {len(bad_excluded)} event(s) with EXCLUDED_CUI (showing up to 10):")
        for name, cui in bad_excluded[:10]:
            print(f"    {name}  entity_cui={cui}")
    else:
        print("  OK - no EXCLUDED_CUIS in events")
    print()

if any_failure:
    raise AssertionError("Forbidden-CUI validation failed (see above).")
print("All output dirs passed forbidden-CUI check.")

In [ ]:
# Events AFTER first clinical contact per horizon output dir.
# For the 'first_contact' horizon this should be 0.

if not first_contact_by_patient:
    print("No first-contact lookup loaded - skipping.")
else:
    for opt in HORIZON_OPTIONS:
        d = output_dir_for(opt)
        if not d.exists():
            print(f"[{label_for(opt)}] missing: {d}")
            continue
        files = sorted(d.glob("*.json"))
        if not files:
            print(f"[{label_for(opt)}] no JSON in {d}")
            continue

        total_n = 0
        total_after = 0
        covered = 0
        missing = 0
        for fpath in files:
            with open(fpath, "r") as f:
                timeline = json.load(f)
            pid = str(timeline.get("patient_id"))
            fc_str = first_contact_by_patient.get(pid)
            if not fc_str:
                missing += 1
                continue
            fc = parse_iso_date(fc_str)
            dated = [e for e in timeline.get("events", []) if e.get("standardized_date")]
            if not dated:
                continue
            covered += 1
            n = len(dated)
            a = sum(1 for e in dated if parse_iso_date(e["standardized_date"]) > fc)
            total_n += n
            total_after += a

        pct = (100.0 * total_after / total_n) if total_n else 0.0
        print(f"[{label_for(opt)}] dir={d}")
        print(f"  #timelines: {len(files)}  covered={covered}  missing_lookup={missing}")
        print(f"  events after first contact: {total_after}/{total_n}  ({pct:.2f}%)")
        print()

In [ ]:
# Boxplot of events per timeline across horizons (no outliers for readability).

labels = []
data_series = []
for opt in HORIZON_OPTIONS:
    d = output_dir_for(opt)
    if not d.exists():
        continue
    lengths = []
    for fpath in sorted(d.glob("*.json")):
        with open(fpath, "r") as f:
            timeline = json.load(f)
        lengths.append(len(timeline.get("events", [])))
    if lengths:
        labels.append(label_for(opt))
        data_series.append(lengths)

if data_series:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.boxplot(data_series, tick_labels=labels, showfliers=False)
    ax.set_ylabel("Number of events per timeline")
    ax.set_title(f"Timeline length by horizon - data={DATA}, cohort={COHORT}")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")